In [ ]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import re

def parse_naver_datetime(date_str):
    m = re.match(r"(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d+):(\d+)", str(date_str))
    if not m:
        return None
    year, month, day, ampm, hour, minute = m.groups()
    hour = int(hour)
    minute = int(minute)
    if ampm == '오후' and hour != 12:
        hour += 12
    if ampm == '오전' and hour == 12:
        hour = 0
    try:
        return datetime(int(year), int(month), int(day), hour, minute)
    except:
        return None

def crawl_multiple_cats_custom_clicks(cat_to_more_clicks, days_ago=2):
    all_results = []
    cutoff_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=days_ago)
    print(f"[INFO] {days_ago}일 전({cutoff_time}) 기사 나오면 섹션별 크롤링 중단")

    for cat, more_clicks in cat_to_more_clicks.items():
        print(f"\n[SECTION {cat}] 더보기 {more_clicks}번 클릭 시작")
        base_url = f'https://news.naver.com/section/{cat}'
        driver = webdriver.Chrome()
        driver.get(base_url)
        time.sleep(1)

        for i in range(more_clicks):
            try:
                more_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, '_CONTENT_LIST_LOAD_MORE_BUTTON'))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                time.sleep(0.2)
                ActionChains(driver).move_to_element(more_button).perform()
                time.sleep(0.1)
                more_button.click()
                print(f"[{cat}] 더보기 {i+1}번 클릭")
                time.sleep(1)
            except Exception as e:
                print(f'[{cat}] 더보기 버튼 클릭 실패:', e)
                break

        # 기사 크롤링
        html = driver.page_source
        soup = BeautifulSoup(html, 'lxml')
        articles = soup.select('a.sa_text_title')
        print(f"[{cat}] 총 기사 수(목록): {len(articles)}")

        visited_urls = set()
        stop_flag = False
        for idx, article in enumerate(articles):
            article_url = article['href']
            if article_url in visited_urls:
                continue
            visited_urls.add(article_url)

            driver.get(article_url)
            time.sleep(0.5)
            article_soup = BeautifulSoup(driver.page_source, 'lxml')

            # 제목
            title_tag = article_soup.select_one('h2#title_area > span')
            title_text = title_tag.get_text(strip=True) if title_tag else '제목 없음'
            # 본문
            content_tag = article_soup.select_one('article#dic_area')
            content_text = content_tag.get_text(strip=True) if content_tag else ''
            # 날짜
            date_tag = article_soup.select_one('span._ARTICLE_DATE_TIME')
            date_text = date_tag.get_text(strip=True) if date_tag else ''
            article_dt = parse_naver_datetime(date_text)

            # 2일 전 기사 만나면 섹션 중단
            if article_dt is not None and article_dt < cutoff_time:
                print(f"[{cat}] 2일 전 기사({article_dt}) 발견! 섹션 크롤링 종료")
                stop_flag = True
                break

            all_results.append({
                'cat': cat,
                'date': date_text,
                'title': title_text,
                'content': content_text,
                'url': article_url
            })
        driver.quit()
        print(f"[{cat}] 누적 기사 건수: {len(all_results)}")
        # 섹션별로 중단(전체는 계속)
        if stop_flag:
            continue

    df = pd.DataFrame(all_results)
    today_str = datetime.now().strftime('%y%m%d')
    outname = f'naver_news_{today_str}.csv'
    df.to_csv(outname, index=False, encoding='utf-8-sig')
    print(f"\n==> 전체 CSV 저장 완료: {outname} (총 {len(df)}건)")
    return outname

# ------------- 사용 예시 -------------
cat_to_more_clicks = {
    '101': 15,   # 예) 101(경제)은 15회
    '105': 8,   # 105(IT/과학)는 8회
    '104': 7    # 104(세계)은 5회
}
outname = crawl_multiple_cats_custom_clicks(cat_to_more_clicks, days_ago=2)
